# 🧪 LAB: Manual MLPs for Classification and Regression

In this lab, you will use `PyTorch` to implement manually a multi-layer perceptron (MLP) for three different tasks: binary classification, multi-class classification and regression. 

## General instructions to complete in ALL three tasks:

1. ***IMPLEMENTATION***:
   
Implement a separate class for each task:

  - `BinaryMLP` for the binary classification task  
  - `MultiClassMLP` for the multi-class classification task  
  - `RegressionMLP` for the regression task

   Each class must include the following methods:

  - `__init__` for initializing 1 or 2 hidden layers. (*Help*: having a second hidden layer only means the need of having a extra set of weights and intercept). 
  - `forward` to transfer information from the input to the output layer. (*Help*: make sure that information passes through a second hidden layer if there is one).
  - `cost` computing the cost.
  - `fit` for training, using the gradients to manually update the parameters of the model. **Use stochastic gradient descent to update your weights**. *N.B.* You may probably reuse much of the code we used in this week tutorial already. 
  - `predict` to convert the information at the output layer into the required output.

2. ***DATA PREPARATION***

For each task, you will be provided with a toy dataset. For each dataset:

  - Split into training and holdout sets (use an 80/20 split)
  - Standardize the features properly, avoiding data leakage
  - Convert all data into `PyTorch` Tensors for compatibility

3. ***MODEL TRAINING AND EVALUATION***

Instantiate the model for each task and train it under different hyperparameter configurations. In each case, record the performance on **both** the training and holdout sets using the appropriate metric for the task (accuracy, MSE, etc.). You should explore the following configurations:

  - One hidden layer, varying the number of hidden units (use ReLU as their activation function)
  - Same number of hidden units across one or two hidden layers (use ReLU as their activation function)
  - Repeat the above setups using Tanh activation instead of ReLU

Present your results in a compact way (e.g., a summary table, a data frame etc).

**NOTE**: When training your model, use a fixed learning rate of your choice (e.g., 0.01 is a reasonable starting point) and a reasonably large number of  epochs (e.g., 100–200) based on how training and holdout performance evolve.

4. ***REFLECTION AND DISCUSSION***

Reflect on the impact of the different hyperparameter settings:

- How does the number of hidden units affect performance?
- What changes when using two layers instead of one?
- How does the activation function (ReLU vs. Tanh) influence results?

Please elaborate your answers.

---

**Collaboration Note**: This assignment is designed to support collaborative work. We encourage you to divide tasks among group members so that everyone can contribute meaningfully. Many components of the assignment can be approached in parallel or split logically across team members. Good coordination and thoughtful integration of your work will lead to a stronger final result.

**Ideally, each group member should be responsible for one of the separate tasks.** BUT, everyone should help each other along the way, both reviewing and refining results and discussion.

---

In total, this lab assignment will be worth **100 points**.

--- 
**Submission notes**:

* Write down all group members' names, or at least the group name (if you have one and you previously provided it), in the first cell of the notebook.

* Verify that the notebook runs as expected and that all required outputs are included.


In [ ]:
NAME(s) = ""

## 1. Pre-implementation Group Discussion (15 points)

Discuss and agree on:

- What cost function should be used for each of the below task.
- What changes are needed in the output layer for each of these tasks. In particular, consider the number of units and the activation function.
- Why it is important to standardize the data before training each model.
- How you could detect overfitting when training your models.

USE AS MANY MARKDOWN CELLS AS NEEDED

## 2. Binary Classification (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

In [ ]:
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)

In [ ]:
# USE AS MANY CELLS AS NEEDED

## 2. Multi-Class Classification (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

---
**NOTE**: This will likely be the most challenging exercise. To help you, here are some pointers:

 - The **output (last) layer** should have as many units as there are classes in your data.  

 - The **activation function of the output layer must be softmax**, not sigmoid. Softmax ensures that all output values are between 0 and 1 and sum to 1, so they can be interpreted as probabilities across the different classes. (*Help*: you may want to look into `torch.nn.functional` to find its implementation...)

 - For a multi-class problem, your target variable `y` should be **one-hot encoded**. For example:  
   - Label = 0 → [1, 0, 0]  
   - Label = 1 → [0, 1, 0]  
   - Label = 2 → [0, 0, 1]  
   You can easily achieve this with `OneHotEncoder` from `sklearn.preprocessing`.  

 - The **predicted class** corresponds to the unit with the highest probability.  
   Example:  
   - `[0.1, 0.3, 0.6] → class 2`  
   - `[0.6, 0.2, 0.2] → class 0`  

 - For this exercise, you need to **implement the categorical cross-entropy loss** (an extension of binary cross-entropy to multiple classes). It is defined as:  

   $$-\sum_{c=1}^{l} y_{o,c}\,\log(p_{o,c}),$$

   where $l$ is the number of classes, $y_{o,c}$ is the one-hot encoded label for observation $o$, and $p_{o,c}$ is the predicted probability for class $c$ (after applying softmax). The log is the natural logarithm.

---

In [11]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=1000, n_classes=3, 
                           n_clusters_per_class=1, 
                           n_features=2, n_informative=2, n_redundant=0, random_state=1234, flip_y=0.15)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd
import torch
from torch import relu
import torch.nn.functional as F


X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.2, random_state=1234, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_holdout = scaler.transform(X_holdout)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_holdout = torch.tensor(X_holdout, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_holdout = torch.tensor(y_holdout, dtype=torch.long)


class MultiClassMLP(object):

    def __init__(self, input_dim, hidden_dim, output_dim, n_hidden_layers=2, learning_rate=0.01, activation="relu"):

        # first hidden layer runs always
        self.W_1 = torch.randn(input_dim, hidden_dim, requires_grad=True)
        self.b_1 = torch.randn(hidden_dim, requires_grad=True)

        # second hidden layer (only used when n_hidden_layers == 2)
        if n_hidden_layers == 2:
            self.W_2 = torch.randn(hidden_dim, hidden_dim, requires_grad=True)
            self.b_2 = torch.randn(hidden_dim, requires_grad=True)

        # output layer (always)
        self.W_3 = torch.randn(hidden_dim, output_dim, requires_grad=True)
        self.b_3 = torch.randn(output_dim, requires_grad=True)

        self.n_hidden_layers = n_hidden_layers
        self.learning_rate = learning_rate
        self.activation = activation

    def forward(self, X):

        h_1 = torch.matmul(X, self.W_1) + self.b_1

        if self.activation == "relu":
            h_1 = relu(h_1)
        elif self.activation == "tanh":
            h_1 = torch.tanh(h_1)

        h = h_1  # output of the last hidden layer

        if self.n_hidden_layers == 2:
            h_2 = torch.matmul(h_1, self.W_2) + self.b_2

            if self.activation == "relu":
                h_2 = relu(h_2)
            elif self.activation == "tanh":
                h_2 = torch.tanh(h_2)

            h = h_2

        out = torch.matmul(h, self.W_3) + self.b_3

        return out

    def cost(self, out, y):
        return F.cross_entropy(out, y)
        # F.cross_entropy applies softmax internally, so `out` stays as raw scores (logits).
        # don't apply softmax in forward(), or it would be applied twice

    def fit(self, X, y, n_epochs=150):

        for epoch in range(n_epochs):

            randperm = torch.randperm(X.shape[0])

            for ii in randperm:
                x_batch = X[ii].unsqueeze(0) # leave not later as to why unsqueeze
                y_batch = y[ii].unsqueeze(0)

                out = self.forward(x_batch)

                cost = self.cost(out, y_batch)

                cost.backward()

                with torch.no_grad(): # no gradient calculation for parameter updates
                    self.W_1 -= self.learning_rate * self.W_1.grad
                    self.b_1 -= self.learning_rate * self.b_1.grad

                    if self.n_hidden_layers == 2:
                        self.W_2 -= self.learning_rate * self.W_2.grad
                        self.b_2 -= self.learning_rate * self.b_2.grad

                    self.W_3 -= self.learning_rate * self.W_3.grad
                    self.b_3 -= self.learning_rate * self.b_3.grad

                    # zero gradients for next iteration
                    self.W_1.grad.zero_()
                    self.b_1.grad.zero_()

                    if self.n_hidden_layers == 2:
                        self.W_2.grad.zero_()
                        self.b_2.grad.zero_()

                    self.W_3.grad.zero_()
                    self.b_3.grad.zero_()

            if epoch % 10 == 0:
                print(f"Epoch {epoch}, Cost: {cost.item():.4f}")

        return self

    def predict(self, X):

        with torch.no_grad(): # disable gradient calculation for prediction
            out = self.forward(X)
            predicted = torch.argmax(out, dim=1) # argmax b/c we want the class with the highest score

        return predicted

In [ ]:
results = []
activations = ["relu", "tanh"]
hidden_layers = [1, 2]
hidden_dims = [8, 16, 32]

# fix this tomorrow to fit the description better, pretty good though
for activation in activations:
    for n_hidden_layers in hidden_layers:
        for hidden_dim in hidden_dims:

            torch.manual_seed(1234)

            model = MultiClassMLP(
                input_dim=X_train.shape[1],
                hidden_dim=hidden_dim,
                output_dim=3,
                n_hidden_layers=n_hidden_layers,
                learning_rate=0.01,
                activation=activation
            )

            model.fit(X_train, y_train, n_epochs=150)

            with torch.no_grad():
                train_cost = model.cost(model.forward(X_train), y_train).item()
                holdout_cost = model.cost(model.forward(X_holdout), y_holdout).item()

            results.append({
                "Hidden layers": n_hidden_layers,
                "Neurons per layer": hidden_dim,
                "Activation": activation,
                "Train Accuracy": accuracy_score(y_train, model.predict(X_train)),
                "Holdout Accuracy": accuracy_score(y_holdout, model.predict(X_holdout)),
                "Train cost": train_cost,
                "Holdout cost": holdout_cost
            })

Epoch 0, Cost: 1.9485
Epoch 10, Cost: 0.4357
Epoch 20, Cost: 1.1986
Epoch 30, Cost: 0.0465
Epoch 40, Cost: 0.0840
Epoch 50, Cost: 0.3524
Epoch 60, Cost: 1.8309
Epoch 70, Cost: 3.5549
Epoch 80, Cost: 0.0777
Epoch 90, Cost: 0.1593
Epoch 100, Cost: 0.1378
Epoch 110, Cost: 0.5206
Epoch 120, Cost: 0.0811
Epoch 130, Cost: 0.2064
Epoch 140, Cost: 0.0465
Epoch 0, Cost: 0.1108
Epoch 10, Cost: 3.0390
Epoch 20, Cost: 0.2127
Epoch 30, Cost: 0.0218
Epoch 40, Cost: 3.5047
Epoch 50, Cost: 0.1051
Epoch 60, Cost: 3.1760
Epoch 70, Cost: 0.1175
Epoch 80, Cost: 0.2588
Epoch 90, Cost: 0.1117
Epoch 100, Cost: 0.0857
Epoch 110, Cost: 0.0985
Epoch 120, Cost: 0.1695
Epoch 130, Cost: 0.1546
Epoch 140, Cost: 0.0902
Epoch 0, Cost: 0.1461
Epoch 10, Cost: 0.0306
Epoch 20, Cost: 0.4892
Epoch 30, Cost: 1.1631
Epoch 40, Cost: 0.2400
Epoch 50, Cost: 0.0630
Epoch 60, Cost: 0.1473
Epoch 70, Cost: 0.1067
Epoch 80, Cost: 0.0987
Epoch 90, Cost: 0.5631
Epoch 100, Cost: 0.0308
Epoch 110, Cost: 0.3552
Epoch 120, Cost: 1.5037
E

In [14]:
results_df = pd.DataFrame(results).round(4)
print(results_df.to_string(index=False))

 Hidden layers  Neurons per layer Activation  Train cost  Holdout cost
             1                  8       relu      0.5494        0.5867
             1                 16       relu      0.5532        0.6343
             1                 32       relu      0.5635        0.6797
             2                  8       relu      0.5238        0.5950
             2                 16       relu      0.5076        0.6024
             2                 32       relu      0.5028        0.6822
             1                  8       tanh      0.5275        0.6110
             1                 16       tanh      0.5227        0.6153
             1                 32       tanh      0.5378        0.6473
             2                  8       tanh      0.5137        0.6380
             2                 16       tanh      0.4907        0.6264
             2                 32       tanh      0.4486        0.6714


## 3: Regression Task (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

In [ ]:
from sklearn.datasets import make_regression
X, y = make_regression(n_samples=1000, n_features=2, n_informative=2, random_state=1234, noise=75)

In [ ]:
# USE AS MANY CELLS AS NEEDED

## 4. Discussion (5 points)

You created a separate class for each task and likely repeated much of the same code across implementations.

Discuss within your group how could you have leveraged inheritance to make your code more reusable and avoid duplication. Provide examples. Be specific.

YOUR TEXT HERE

## 5. Collaboration Reflection (5 points)

As a group, identify **one specific challenge or inefficiency** you encountered while working on this lab. In 3–5 sentences:

1. Briefly describe what happened and how it affected your work.
2. Identify one concrete action your group will take to avoid or better manage this issue in the next lab.

If your group encountered no major problem, identify one aspect of your collaboration that could still be improved.

YOUR TEXT HERE